In [ ]:
import SimpleITK as sitk
import os

def resample_image(image, reference_image, is_label=False):
    """
    Resample an image or segmentation mask.

    Parameters:
    - image: The image to be resampled (SimpleITK Image object)
    - reference_image: The reference image defining the target space (SimpleITK Image object)
    - is_label: Indicates if the image is a segmentation mask (boolean), determines the interpolation method

    Returns:
    - Resampled image (SimpleITK Image object)
    """
    # Define the resampler
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(reference_image)
    resampler.SetOutputSpacing(reference_image.GetSpacing())
    resampler.SetSize(reference_image.GetSize())
    resampler.SetOutputDirection(reference_image.GetDirection())
    resampler.SetOutputOrigin(reference_image.GetOrigin())
    resampler.SetTransform(sitk.Transform())

    # Set the interpolation method
    if is_label:
        # Use nearest neighbor interpolation for segmentation masks
        resampler.SetInterpolator(sitk.sitkNearestNeighbor)
    else:
        # Use linear interpolation or other suitable methods for images
        resampler.SetInterpolator(sitk.sitkLinear)

    # Perform resampling
    resampled_image = resampler.Execute(image)
    return resampled_image

# Example usage
# Read the original image and segmentation mask
original_image = sitk.ReadImage(r'/path/to/workspace/classification_model_originalimage/train_data/SUBJECT_ID_REDACTED_4_ax_t2_frfse_ardl_h_1nex.nii.gz')
original_mask1 = sitk.ReadImage(r'/path/to/workspace/classification_model_originalimage/totalsegment_work/train/SUBJECT_ID_REDACTED_4_ax_t2_frfse_ardl_h_1nex/prostate.nii.gz')
original_mask2 = sitk.ReadImage(r'/path/to/workspace/classification_model_originalimage/totalsegment_work/train/SUBJECT_ID_REDACTED_4_ax_t2_frfse_ardl_h_1nex/colon.nii.gz')


# Define the target spacing
new_spacing = (0.39, 0.39, 3.3)  # Adjust as needed

# Compute the new size
original_size = original_image.GetSize()
original_spacing = original_image.GetSpacing()
new_size = [
    int(round(original_size[i] * (original_spacing[i] / new_spacing[i])))
    for i in range(3)
]

# Create a reference image (contains only spatial information)
reference_image = sitk.Image(new_size, original_image.GetPixelID())
reference_image.SetSpacing(new_spacing)
reference_image.SetOrigin(original_image.GetOrigin())
reference_image.SetDirection(original_image.GetDirection())

# Resample the original image
resampled_image = resample_image(original_image, reference_image, is_label=False)

# Resample the segmentation mask
resampled_mask1 = resample_image(original_mask1, reference_image, is_label=True)
resampled_mask2 = resample_image(original_mask2, reference_image, is_label=True)

save_path = r'/path/to/workspace/classification_model_originalimage/totalsegment_work_resample/male'
os.makedirs(save_path, exist_ok=True)

save_path1 = os.path.join(save_path, 'resampled_image.nii.gz')
save_path2 = os.path.join(save_path, 'resampled_mask1.nii.gz')
save_path3 = os.path.join(save_path, 'resampled_mask2.nii.gz')

# Save the resampled image and mask
sitk.WriteImage(resampled_image, save_path1)
sitk.WriteImage(resampled_mask1, save_path2)
sitk.WriteImage(resampled_mask2, save_path3)

print('done')

In [ ]:
import SimpleITK as sitk
import numpy as np

def extract_rectum(colon_mask_path, prostate_mask_path, output_rectum_mask_path):
    """
    Extracts the rectum region from the colon mask using the prostate mask as a reference.

    Parameters:
    - colon_mask_path: Path to the colon mask NIfTI file (.nii or .nii.gz)
    - prostate_mask_path: Path to the prostate mask NIfTI file (.nii or .nii.gz)
    - output_rectum_mask_path: Path to save the extracted rectum mask NIfTI file
    """
    # Load the colon and prostate masks
    colon_mask = sitk.ReadImage(colon_mask_path)
    prostate_mask = sitk.ReadImage(prostate_mask_path)
    
    # Ensure both masks have the same spatial information
    assert colon_mask.GetSize() == prostate_mask.GetSize(), "Masks must have the same dimensions"
    assert colon_mask.GetSpacing() == prostate_mask.GetSpacing(), "Masks must have the same spacing"
    assert colon_mask.GetOrigin() == prostate_mask.GetOrigin(), "Masks must have the same origin"
    assert colon_mask.GetDirection() == prostate_mask.GetDirection(), "Masks must have the same direction"
    
    # Convert masks to numpy arrays for processing
    colon_array = sitk.GetArrayFromImage(colon_mask)
    prostate_array = sitk.GetArrayFromImage(prostate_mask)
    print('colon shape', colon_array.shape)
    print('no-zeros number area', np.count_nonzero(colon_array))
    
    
    # Ensure binary masks (values are 0 or 1)
    colon_array = (colon_array > 0).astype(np.uint8)
    prostate_array = (prostate_array > 0).astype(np.uint8)
    
    # Label connected components in the colon mask
    connected_component_filter = sitk.ConnectedComponentImageFilter()
    connected_component_filter.SetFullyConnected(True)
    colon_cc = connected_component_filter.Execute(sitk.GetImageFromArray(colon_array))
    sitk.WriteImage(colon_cc, '/path/to/workspace/classification_model_originalimage/totalsegment_work_resample/male/colon_cc.nii.gz')
    
    # Relabel components to get unique labels
    relabel_filter = sitk.RelabelComponentImageFilter()
    colon_cc = relabel_filter.Execute(colon_cc)
    
    # Convert labeled colon components to numpy array
    colon_cc_array = sitk.GetArrayFromImage(colon_cc)
    num_components = relabel_filter.GetNumberOfObjects()
    print(num_components)
    
    # Initialize an empty array for the rectum mask
    rectum_array = np.zeros_like(colon_array)
    
    # Get the prostate mask bounding box
    prostate_indices = np.argwhere(prostate_array > 0)
    prostate_min_y = np.min(prostate_indices[:, 1])  # Assuming y-axis is the first axis
    prostate_max_y = np.max(prostate_indices[:, 1])
    
    # For each connected component in the colon mask
    for label in range(1, num_components + 1):
        # Extract the component
        component_array = (colon_cc_array == label).astype(np.uint8)
        
        # Get the bounding box of the component
        component_indices = np.argwhere(component_array > 0)
        component_min_y = np.min(component_indices[:, 1])
        component_max_y = np.max(component_indices[:, 1])
        
        # Check if the component is below the prostate mask
        if component_max_y < prostate_min_y:
            # Assuming the rectum is approximately circular, you can add shape analysis here if needed
            # For now, select the component
            rectum_array += component_array  # Combine if multiple components satisfy the condition
        else:
            print('nothing')
    
    # Convert the rectum array back to a SimpleITK image
    rectum_mask = sitk.GetImageFromArray(rectum_array)
    rectum_mask.CopyInformation(colon_mask)
    
    # Save the rectum mask
    sitk.WriteImage(rectum_mask, output_rectum_mask_path)
    print(f"Rectum mask saved to {output_rectum_mask_path}")


colon_mask_path = r'/path/to/workspace/classification_model_originalimage/totalsegment_work_resample/male/resampled_mask2.nii.gz'
prostate_mask_path = r'/path/to/workspace/classification_model_originalimage/totalsegment_work_resample/male/resampled_mask1.nii.gz'
output_rectum_mask_path = r'/path/to/workspace/classification_model_originalimage/totalsegment_work_resample/male/rectum.nii.gz'

# Extract the rectum mask
extract_rectum(colon_mask_path, prostate_mask_path, output_rectum_mask_path)